# Chapter 5 · The full run

Chapter 4's single block reached 1.76. A full GPT stacks several blocks. Each one is
attention followed by a small feed-forward network, with two additions that make tall
stacks trainable: *residual connections* and *layer normalisation*.

In this chapter you train the full tinyGPT: six blocks, 10.7 million parameters, in under
ten minutes.

| | |
|---|---|
| **Open box** | one transformer block: `Block` in `tinygpt/model.py` |
| **The number** | 1.76 → 1.49 |
| **Time** | about an hour, including one training run of 6 to 8 minutes |
| **Python you need** | reading a class; dictionaries |
| **GPU** | needed. In Colab: Runtime > Change runtime type > T4 GPU |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    repo = Path("/content/tinyGPT")
    if not repo.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(repo))
    root = repo / "entry"
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

## 1 · Predict `everyone`

tinyGPT has 6 blocks, 6 heads in each, 384 numbers per character and 128 characters of
memory. It trains for 2,500 steps.

In [ ]:
guess_loss = None       # its validation loss at the end? (one block: 1.76)
guess_step = None       # at which step will it beat counting pairs (2.48)?
guess_minutes = None    # how many minutes will training take on your machine?

## 2 · Run `everyone`

Here are its settings, from `configs/base.yaml`:

In [ ]:
import torch

tok, train_ids, val_ids = tinygpt.load_data()
cfg = tinygpt.load_config("configs/base.yaml")
print(cfg)

Start training now. On a Colab T4 it should take about 6 to 8 minutes; the README lists
estimates for other machines. If your machine is slower, training stops by itself after 9 minutes with a
higher loss. While it runs, read the questions in section 3.

In [ ]:
model, history = tinygpt.train(cfg, save_path="runs/base/model.pt")

In [ ]:
tinygpt.plot_history(history)
print("your guesses:", guess_loss, guess_step, guess_minutes)

Let it write. This is temperature 1.0: the model's own probabilities, not tidied up.

In [ ]:
print(tinygpt.sample(model, tok, prompt="ROMEO:", length=500, seed=0))

### Keep your model

Chapter 6 uses the model you just trained. On Colab, everything disappears when the
session ends, so this copies the model to your Google Drive; Colab will ask for
permission the first time. On your own computer it is already saved.

In [ ]:
tinygpt.keep_model()

## 3 · Investigate `everyone`

### Open the box

One block. tinyGPT stacks six of them.

In [ ]:
tinygpt.show(tinygpt.model.Block)

1. **Residual connections.** `x = x + self.attn(...)` *adds* the block's result to what
   came in, instead of replacing it. Each block makes a correction rather than starting
   over. Why might that make it easier to stack six blocks than if each one replaced `x`?
2. **Layer normalisation.** `ln1` and `ln2` rescale each character's numbers to a steady
   range before attention and before the feed-forward network. What could go wrong, six
   blocks deep, without that?
3. **Attention, then feed-forward.** Attention moves information *between* positions;
   the feed-forward network works on each position *on its own*. Look at the plot: when
   did the model pass 2.48? 2.0?
4. **Training versus validation.** At the end, training loss is well below validation
   loss. What has the model started to do?

*Your explanation (double-click to edit):*

### Where do the 10.7 million numbers live?

Add up the parameters by part, across all six blocks:

In [ ]:
sizes = {}
for name, p in model.named_parameters():
    parts = name.split(".")                                    # e.g. blocks.3.ffwd.net.0.weight
    part = parts[2] if parts[0] == "blocks" else parts[0]
    sizes[part] = sizes.get(part, 0) + p.numel()
for part, n in sizes.items():
    print(f"{part:<10}{n:>12,}")
print(f"{'total':<10}{tinygpt.count_params(model):>12,}")

Which part holds most of the model? Is that what you expected from the name *attention* model?

*Your explanation (double-click to edit):*

## 4 · Modify `most people`

Each change below uses the `small` model, which trains in about a minute on a GPU, so
you can compare one change at a time. **Predict before each run.**

**A. How many blocks?** One, two and four.

In [ ]:
blocks = {}
for n in [1, 2, 4]:
    _, blocks[f"{n} block{'s' if n > 1 else ''}"] = tinygpt.train(tinygpt.load_config("configs/small.yaml", n_layer=n))
tinygpt.plot_history(blocks)

**B. No sense of position.** `pos_emb=False` removes the table that tells the model
*where* each character is. Attention alone sees which characters are in the window, but
not their order.

In [ ]:
_, no_position = tinygpt.train(tinygpt.load_config("configs/small.yaml", pos_emb=False))
tinygpt.plot_history({"with positions": blocks["4 blocks"], "without positions": no_position})

**C. Dropout.** While training, dropout switches off a random 20% of the numbers inside
the model at every step, so it can't lean too hard on any one of them. Watch the gap
between training and validation loss.

In [ ]:
_, with_dropout = tinygpt.train(tinygpt.load_config("configs/small.yaml", dropout=0.2))
print("without dropout, the last row:", blocks["4 blocks"][-1])
print("with dropout,    the last row:", with_dropout[-1])

Did dropout help this small model, or hurt it? The full tinyGPT uses dropout 0.2: why
might a model with 13 times as many parameters need it more?

Which change hurt most? Which barely mattered at this size? Explain one result:

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

**Part 1. Count the parameters by hand.** Write a formula for the exact number of
parameters in a GPT, using what you know about its parts:

- `nn.Embedding(a, b)` holds a × b numbers.
- `nn.Linear(a, b)` holds a × b weights plus b biases, unless it says `bias=False`.
- `nn.LayerNorm(d)` holds 2 × d numbers: a scale and a shift for each of the d.
- A head's `query`, `key` and `value` are each `Linear(n_embd, head_size, bias=False)`,
  and a block has `n_head` heads, where `n_head × head_size = n_embd`.
- After the heads, attention has a `Linear(n_embd, n_embd)`. The feed-forward network is
  `Linear(n_embd, 4 * n_embd)` then `Linear(4 * n_embd, n_embd)`.
- Around the blocks: `tok_emb` (`Embedding(vocab_size, n_embd)`), `pos_emb`
  (`Embedding(block_size, n_embd)`), `ln_f` and `lm_head` (`Linear(n_embd, vocab_size)`).

In [ ]:
def param_count(vocab_size, block_size, n_layer, n_embd):
    """The exact number of parameters in a GPT with these settings."""
    d = n_embd
    embeddings = 0                # tok_emb + pos_emb
    per_block = 0                 # ln1 + attention (heads and proj) + ln2 + feed-forward
    final = 0                     # ln_f + lm_head
    return embeddings + n_layer * per_block + final

tinygpt.check.param_count(param_count)

Notice that `n_head` isn't one of the inputs. Why doesn't the number of heads change the
number of parameters?

**Part 2. The three-minute race.** Change the settings below to get the lowest
validation loss you can in three minutes of training. Compare only with people on the
same kind of machine: a faster GPU gets more steps in three minutes.

In [ ]:
race = tinygpt.load_config("configs/small.yaml",
                           n_layer=4, n_head=4, n_embd=128, block_size=64,     # try changing these
                           dropout=0.0, learning_rate=0.001, batch_size=32,
                           max_iters=100_000, max_minutes=3.0)                 # keep: three minutes each
_, race_history = tinygpt.train(race)
print(f"best validation loss in three minutes: {min(r['val_loss'] for r in race_history):.4f}")

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/solutions/05_full_run.md) (also in the `solutions/` folder of your copy).

## Where the number stands

In [ ]:
tinygpt.scoreboard({"one block (chapter 4)": 1.76, "tinyGPT, yours": history[-1]["val_loss"]})

## Common questions

**Why does validation loss level off while training loss keeps falling?** The model has
started to memorise its 1 million training characters rather than learn patterns that
carry over to new text. Dropout slows that down; more training text would help most.

**Would training for longer help?** Not much. In our measured run, validation loss went
from 1.51 at step 1,750 to 1.49 at step 2,500, while training loss kept dropping.

**How does this compare with GPT-2 or ChatGPT?** Same design: a stack of these blocks.
GPT-2 small has 12 blocks of 768 numbers per token and 124 million parameters, trained on
about 40 GB of text; ours has 10.7 million, trained on 1 MB. Today's chat models are far
bigger again, and trained on far more.

**My run stopped early and my number is higher. Did I do something wrong?** No: your
machine is slower than a T4, and every run stops after 9 minutes. Use Colab with a T4 GPU
to reach 1.49.

## If something breaks

| What you see | What to do |
|---|---|
| `warning: no GPU found` and very slow training | Switch on a GPU (Colab: Runtime > Change runtime type > T4 GPU), then run the setup cell again. |
| `stopped early at step ...` | Your machine is slower than a T4; see the question above. |
| `CUDA out of memory` | Restart the runtime, run the setup cell, and add `batch_size=32` to `load_config(...)`. |
| The Colab session disconnects mid-run | Run the setup cell again, then the cells from section 2. |
| `keep_model` can't reach Google Drive | Run the cell again and allow access, with the Google account you use for Colab. |

## Next

**Chapter 6 · Talking to your model.** What 1.49 looks like as writing: temperature,
sampling, and whether the model is just copying Shakespeare.